# Yash Gaikwad
# Roll no-381071

In [ ]:
Implement a Federated Learning framework for a linear regression task where multiple clients
collaboratively learn a global prediction model without sharing their raw data. For a house price
dataset can be split across clients, each client trains a local linear regression model, and a central
server applies Federated Averaging (FedAvg) to combine the local model parameters into a
global model while preserving data privacy.

In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_squared_error
from sklearn.metrics import precision_score, recall_score, f1_score

In [4]:
data = pd.read_csv("Housing.csv")

print("Dataset Shape:", data.shape)
print(data.head())

Dataset Shape: (545, 13)
      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  


In [5]:
encoder = LabelEncoder()

categorical_cols = [
    'mainroad','guestroom','basement',
    'hotwaterheating','airconditioning',
    'prefarea','furnishingstatus'
]

for col in categorical_cols:
    data[col] = encoder.fit_transform(data[col])

In [6]:
X = data.drop("price", axis=1)
y = data["price"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
NUM_CLIENTS = 5

X_clients = np.array_split(X_train, NUM_CLIENTS)
y_clients = np.array_split(y_train, NUM_CLIENTS)

print("Clients Created:", NUM_CLIENTS)

Clients Created: 5


c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)


In [9]:
def fed_avg(weights, biases):
    avg_weights = np.mean(weights, axis=0)
    avg_bias = np.mean(biases)
    return avg_weights, avg_bias

In [10]:
ROUNDS = 5

global_weights = np.zeros(X.shape[1])
global_bias = 0

for r in range(ROUNDS):

    local_weights = []
    local_biases = []

    for i in range(NUM_CLIENTS):
        model = LinearRegression()
        model.coef_ = global_weights
        model.intercept_ = global_bias

        model.fit(X_clients[i], y_clients[i])

        local_weights.append(model.coef_)
        local_biases.append(model.intercept_)

    global_weights, global_bias = fed_avg(local_weights, local_biases)
    print(f"Round {r+1} Completed")

Round 1 Completed
Round 2 Completed
Round 3 Completed
Round 4 Completed
Round 5 Completed


In [11]:
y_pred = np.dot(X_test, global_weights) + global_bias

In [12]:
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)

Mean Squared Error: 1797977505939.4106


In [13]:
threshold = np.median(y_test)

y_test_binary = (y_test > threshold).astype(int)
y_pred_binary = (y_pred > threshold).astype(int)

In [14]:
precision = precision_score(y_test_binary, y_pred_binary)
recall = recall_score(y_test_binary, y_pred_binary)
f1 = f1_score(y_test_binary, y_pred_binary)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.8148148148148148
Recall: 0.8148148148148148
F1 Score: 0.8148148148148148
